# Week 67, Day 3: Async I/O patterns

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w067d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def bounded(delays, limit):
    "Start each job, in order, when one of `limit` slots is free."
    free_at = [0] * limit
    finish = []
    for d in delays:
        i = free_at.index(min(free_at))
        free_at[i] += d
        finish.append(free_at[i])
    return finish

delays = [2, 2, 2, 2, 2, 2]
print(bounded(delays, 6), bounded(delays, 2))
print(max(bounded(delays, 6)), max(bounded(delays, 2)))

In [ ]:
def peak_concurrency(starts_and_ends):
    "The most intervals open at once."
    events = sorted([(s, 1) for s, e in starts_and_ends] + [(e, -1) for s, e in starts_and_ends])
    open_now = peak = 0
    for _, change in events:
        open_now += change
        peak = max(peak, open_now)
    return peak

unlimited = [(0, 2)] * 6
limited = [(0, 2), (0, 2), (2, 4), (2, 4), (4, 6), (4, 6)]
print(peak_concurrency(unlimited), peak_concurrency(limited))

In [ ]:
from collections import deque

def simulate_queue(events):
    "events: ('put', item) or ('get', name). Waiting getters are served in order."
    items, waiting, served = deque(), deque(), []
    for kind, value in events:
        if kind == "put":
            if waiting:
                served.append((waiting.popleft(), value))
            else:
                items.append(value)
        elif items:
            served.append((value, items.popleft()))
        else:
            waiting.append(value)
    return served, list(waiting)

print(simulate_queue([("get", "w1"), ("get", "w2"), ("put", "a"), ("put", "b"), ("put", "c"), ("get", "w3")]))

In [ ]:
import random

def backoff(attempts, base=1, factor=2, cap=30, jitter=0.0, rng=None):
    rng = rng or random.Random(0)
    return [min(cap, base * factor ** i) + jitter * rng.random() for i in range(attempts - 1)]

print(backoff(6))
print([round(x, 2) for x in backoff(4, jitter=1.0, rng=random.Random(42))])
print("total waiting before giving up:", sum(backoff(6)))

In [ ]:
class Pause:
    def __await__(self):
        yield

class Session:
    def __init__(self, log):
        self.log = log
    async def __aenter__(self):
        await Pause()
        self.log.append("open")
        return self
    async def __aexit__(self, exc_type, exc, tb):
        await Pause()
        self.log.append("close")
        return False

async def main(log):
    async with Session(log) as s:
        log.append("work")
    return log

coro = main([])
try:
    while True:
        coro.send(None)
except StopIteration as done:
    print(done.value)

In [ ]:
class Pause:
    def __await__(self):
        yield

async def numbers(n):
    for i in range(n):
        await Pause()
        yield i

async def evens(stream):
    async for x in stream:
        if x % 2 == 0:
            yield x

async def main():
    return [x * 10 async for x in evens(numbers(7))]

coro = main()
try:
    while True:
        coro.send(None)
except StopIteration as done:
    print(done.value)

> **Key idea.** Limit concurrency with asyncio.Semaphore(n) and async with around each operation. Pass work between tasks with asyncio.Queue: put, get (waiting while empty), a fixed pool of consumers and one sentinel per consumer; inside, put wakes one waiting getter, which re-checks the queue. Put deadlines on I/O with wait_for or asyncio.timeout, and retry temporary failures with exponential backoff (capped, with jitter) only when the operation is safe to repeat. Async context managers await __aenter__ and __aexit__ (or use asynccontextmanager) for set-up and clean-up that waits. Async generators produce streams consumed with async for and async comprehensions, and compose like ordinary generators.

### Exercise: Bounded concurrency in numbers

Write `bounded_finish(delays, limit)`: the finish time of each job when jobs start in order, each as soon as one of `limit` slots is free (the slot free first; the lowest-numbered on ties), and `peak(intervals)`: the most `(start, end)` intervals open at the same time, where an interval ending at the same moment another starts doesn't overlap it. Then `smallest_limit(delays, deadline)`: the smallest limit (from 1 up to the number of jobs) under which everything finishes by `deadline`, or `None` if even no limit at all is too slow.

In [ ]:
def bounded_finish(delays, limit):
    ___

def peak(intervals):
    ___

def smallest_limit(delays, deadline):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_bounded():\n    \"concurrency limits\"\n    assert bounded_finish([2, 2, 2, 2], 2) == [2, 2, 4, 4] and bounded_finish([5, 1, 1, 1], 2) == [5, 1, 2, 3]\n    assert peak([(0, 2), (0, 2), (2, 4)]) == 2 and peak([]) == 0 and peak([(0, 5), (1, 2), (1, 3)]) == 3\n    assert smallest_limit([2] * 6, 6) == 2 and smallest_limit([2] * 6, 2) == 6 and smallest_limit([5], 4) is None\n    assert smallest_limit([], 1) == 1")

### Exercise: Retrying with backoff

Network calls fail now and then, and the standard cure is to **retry** with **exponential backoff**: wait 1, then 2, then 4 units, and so on, up to a cap, so a struggling server isn't hammered. Using the loop given in the starter, write `backoff_delays(attempts, base=1, factor=2, cap=30)`: the list of waits between `attempts` tries (one fewer than the attempts), and a coroutine `retry(loop, op, attempts, base=1, factor=2, cap=30)` that awaits `op()` (a coroutine function), and on an exception waits the next backoff delay with `Sleep` and tries again, re-raising the last exception if every attempt fails.

In [ ]:
import heapq
from collections import deque

class Sleep:
    def __init__(self, seconds):
        self.seconds = seconds
    def __await__(self):
        yield ("sleep", self.seconds)

class Waitable:
    "Anything tasks can wait on: the loop adds the waiting task to .waiters."
    def __init__(self):
        self.waiters = deque()
    def wait(self):
        return _Wait(self)

class _Wait:
    def __init__(self, target):
        self.target = target
    def __await__(self):
        yield ("wait", self.target)

class Task(Waitable):
    def __init__(self, coro):
        super().__init__()
        self.coro, self.done, self.result = coro, False, None
    def __await__(self):
        if not self.done:
            yield ("wait", self)
        return self.result

class Loop:
    def __init__(self):
        self.now, self.ready, self.sleepers, self._seq = 0, deque(), [], 0
    def create_task(self, coro):
        task = Task(coro)
        self.ready.append(task)
        return task
    def wake(self, waitable, count=None):
        "Move waiting tasks (all, or up to count) back to the ready queue."
        while waitable.waiters and (count is None or count > 0):
            self.ready.append(waitable.waiters.popleft())
            if count is not None:
                count -= 1
    def _step(self, task):
        try:
            kind, arg = task.coro.send(None)
        except StopIteration as stop:
            task.done, task.result = True, stop.value
            self.wake(task)
            return
        if kind == "sleep":
            self._seq += 1
            heapq.heappush(self.sleepers, (self.now + arg, self._seq, task))
        elif kind == "wait":
            arg.waiters.append(task)
    def run_until_complete(self, coro):
        main = self.create_task(coro)
        while self.ready or self.sleepers:
            if not self.ready:
                self.now, _, task = heapq.heappop(self.sleepers)
                self.ready.append(task)
                continue
            self._step(self.ready.popleft())
        return main.result

def backoff_delays(attempts, base=1, factor=2, cap=30):
    ___

async def retry(loop, op, attempts, base=1, factor=2, cap=30):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_retry():\n    \"retries with backoff\"\n    assert backoff_delays(4) == [1, 2, 4] and backoff_delays(6, base=2, cap=10) == [2, 4, 8, 10, 10]\n    assert backoff_delays(1) == []\n    loop = Loop()\n    calls = []\n    async def flaky():\n        calls.append(loop.now)\n        if len(calls) < 3:\n            raise ConnectionError(\"down\")\n        return \"ok\"\n    async def main():\n        return await retry(loop, flaky, attempts=5)\n    assert loop.run_until_complete(main()) == \"ok\" and calls == [0, 1, 3]\n    loop2 = Loop()\n    async def always_fails():\n        raise ConnectionError(\"down\")\n    async def main2():\n        try:\n            await retry(loop2, always_fails, attempts=3, base=5)\n        except ConnectionError:\n            return \"gave up\", loop2.now\n    assert loop2.run_until_complete(main2()) == (\"gave up\", 15)")

### Exercise: An async queue

Write `AsyncQueue(loop)`, a `Waitable` (from the loop code in the starter) holding items: `put(item)` adds an item and wakes **one** waiting task with `loop.wake(self, 1)`; the coroutine `get()` waits (with `await self.wait()`) while the queue is empty, then returns the oldest item. Then a coroutine `pipeline(loop, items, workers, handle)`: start `workers` worker tasks that take items from a queue and record `(worker number, await handle(item))` in a shared list in the order they finish, put the items and then one `None` sentinel per worker, wait for all the workers, and return the list.

In [ ]:
import heapq
from collections import deque

class Sleep:
    def __init__(self, seconds):
        self.seconds = seconds
    def __await__(self):
        yield ("sleep", self.seconds)

class Waitable:
    "Anything tasks can wait on: the loop adds the waiting task to .waiters."
    def __init__(self):
        self.waiters = deque()
    def wait(self):
        return _Wait(self)

class _Wait:
    def __init__(self, target):
        self.target = target
    def __await__(self):
        yield ("wait", self.target)

class Task(Waitable):
    def __init__(self, coro):
        super().__init__()
        self.coro, self.done, self.result = coro, False, None
    def __await__(self):
        if not self.done:
            yield ("wait", self)
        return self.result

class Loop:
    def __init__(self):
        self.now, self.ready, self.sleepers, self._seq = 0, deque(), [], 0
    def create_task(self, coro):
        task = Task(coro)
        self.ready.append(task)
        return task
    def wake(self, waitable, count=None):
        "Move waiting tasks (all, or up to count) back to the ready queue."
        while waitable.waiters and (count is None or count > 0):
            self.ready.append(waitable.waiters.popleft())
            if count is not None:
                count -= 1
    def _step(self, task):
        try:
            kind, arg = task.coro.send(None)
        except StopIteration as stop:
            task.done, task.result = True, stop.value
            self.wake(task)
            return
        if kind == "sleep":
            self._seq += 1
            heapq.heappush(self.sleepers, (self.now + arg, self._seq, task))
        elif kind == "wait":
            arg.waiters.append(task)
    def run_until_complete(self, coro):
        main = self.create_task(coro)
        while self.ready or self.sleepers:
            if not self.ready:
                self.now, _, task = heapq.heappop(self.sleepers)
                self.ready.append(task)
                continue
            self._step(self.ready.popleft())
        return main.result

class AsyncQueue(Waitable):
    def __init__(self, loop):
        ___

async def pipeline(loop, items, workers, handle):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_queue():\n    \"an async queue\"\n    loop = Loop()\n    q = AsyncQueue(loop)\n    got = []\n    async def consumer():\n        for _ in range(2):\n            got.append((await q.get(), loop.now))\n    async def producer():\n        await Sleep(3)\n        q.put(\"a\")\n        await Sleep(2)\n        q.put(\"b\")\n    async def main():\n        c = loop.create_task(consumer())\n        await producer()\n        await c\n    loop.run_until_complete(main())\n    assert got == [(\"a\", 3), (\"b\", 5)]\n    loop2 = Loop()\n    async def handle(x):\n        await Sleep(x)\n        return x * 10\n    async def main2():\n        return await pipeline(loop2, [3, 1, 1, 1], workers=2, handle=handle)\n    results = loop2.run_until_complete(main2())\n    assert sorted(r for _, r in results) == [10, 10, 10, 30] and loop2.now == 3\n    assert results == [(1, 10), (1, 10), (0, 30), (1, 10)]")

### Exercise: An async context manager

Some set-up and clean-up must itself wait: opening a network connection, closing it politely. Classes support `async with` through the coroutines `__aenter__` and `__aexit__`. Write `Connection(log)`: entering waits 1 unit (`await Sleep(1)`), sets `open` and logs `"connected"`; leaving waits 1 unit, clears `open` and logs `"closed"`, or `"closed after KeyError"` (the exception's type name) if the block raised, without suppressing the exception; the coroutine `query(sql)` raises `RuntimeError("not connected")` if it isn't open, otherwise waits 2 units and returns `"rows for " + sql`. Then the coroutine `use(conn, sql)`, which runs one query inside `async with`.

In [ ]:
import heapq
from collections import deque

class Sleep:
    def __init__(self, seconds):
        self.seconds = seconds
    def __await__(self):
        yield ("sleep", self.seconds)

class Waitable:
    "Anything tasks can wait on: the loop adds the waiting task to .waiters."
    def __init__(self):
        self.waiters = deque()
    def wait(self):
        return _Wait(self)

class _Wait:
    def __init__(self, target):
        self.target = target
    def __await__(self):
        yield ("wait", self.target)

class Task(Waitable):
    def __init__(self, coro):
        super().__init__()
        self.coro, self.done, self.result = coro, False, None
    def __await__(self):
        if not self.done:
            yield ("wait", self)
        return self.result

class Loop:
    def __init__(self):
        self.now, self.ready, self.sleepers, self._seq = 0, deque(), [], 0
    def create_task(self, coro):
        task = Task(coro)
        self.ready.append(task)
        return task
    def wake(self, waitable, count=None):
        "Move waiting tasks (all, or up to count) back to the ready queue."
        while waitable.waiters and (count is None or count > 0):
            self.ready.append(waitable.waiters.popleft())
            if count is not None:
                count -= 1
    def _step(self, task):
        try:
            kind, arg = task.coro.send(None)
        except StopIteration as stop:
            task.done, task.result = True, stop.value
            self.wake(task)
            return
        if kind == "sleep":
            self._seq += 1
            heapq.heappush(self.sleepers, (self.now + arg, self._seq, task))
        elif kind == "wait":
            arg.waiters.append(task)
    def run_until_complete(self, coro):
        main = self.create_task(coro)
        while self.ready or self.sleepers:
            if not self.ready:
                self.now, _, task = heapq.heappop(self.sleepers)
                self.ready.append(task)
                continue
            self._step(self.ready.popleft())
        return main.result

class Connection:
    def __init__(self, log):
        ___

async def use(conn, sql):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_async_with():\n    \"async context managers\"\n    loop = Loop()\n    log = []\n    conn = Connection(log)\n    async def main():\n        return await use(conn, \"select 1\"), loop.now\n    assert loop.run_until_complete(main()) == (\"rows for select 1\", 4) and log == [\"connected\", \"closed\"]\n    loop2 = Loop()\n    log2 = []\n    async def bad():\n        async with Connection(log2) as c:\n            raise KeyError(\"x\")\n    async def main2():\n        try:\n            await bad()\n        except KeyError:\n            return \"caught\"\n    assert loop2.run_until_complete(main2()) == \"caught\" and log2 == [\"connected\", \"closed after KeyError\"]\n    loop3 = Loop()\n    async def unconnected():\n        try:\n            await Connection([]).query(\"x\")\n        except RuntimeError as e:\n            return str(e)\n    assert loop3.run_until_complete(unconnected()) == \"not connected\"")

### Exercise: Streams with async generators

Write two async generators using the loop in the starter: `ticks(loop, n, interval)` sleeps `interval` and then yields the current `loop.now`, `n` times; and `rate_limited(loop, items, gap)` yields the items with at least `gap` units between them, measuring from when the previous item was **yielded** (so a slow consumer doesn't cause extra waiting). Then the coroutine `collect(agen)`, returning everything an async generator yields as a list, using an async comprehension (`[x async for x in agen]`).

In [ ]:
import heapq
from collections import deque

class Sleep:
    def __init__(self, seconds):
        self.seconds = seconds
    def __await__(self):
        yield ("sleep", self.seconds)

class Waitable:
    "Anything tasks can wait on: the loop adds the waiting task to .waiters."
    def __init__(self):
        self.waiters = deque()
    def wait(self):
        return _Wait(self)

class _Wait:
    def __init__(self, target):
        self.target = target
    def __await__(self):
        yield ("wait", self.target)

class Task(Waitable):
    def __init__(self, coro):
        super().__init__()
        self.coro, self.done, self.result = coro, False, None
    def __await__(self):
        if not self.done:
            yield ("wait", self)
        return self.result

class Loop:
    def __init__(self):
        self.now, self.ready, self.sleepers, self._seq = 0, deque(), [], 0
    def create_task(self, coro):
        task = Task(coro)
        self.ready.append(task)
        return task
    def wake(self, waitable, count=None):
        "Move waiting tasks (all, or up to count) back to the ready queue."
        while waitable.waiters and (count is None or count > 0):
            self.ready.append(waitable.waiters.popleft())
            if count is not None:
                count -= 1
    def _step(self, task):
        try:
            kind, arg = task.coro.send(None)
        except StopIteration as stop:
            task.done, task.result = True, stop.value
            self.wake(task)
            return
        if kind == "sleep":
            self._seq += 1
            heapq.heappush(self.sleepers, (self.now + arg, self._seq, task))
        elif kind == "wait":
            arg.waiters.append(task)
    def run_until_complete(self, coro):
        main = self.create_task(coro)
        while self.ready or self.sleepers:
            if not self.ready:
                self.now, _, task = heapq.heappop(self.sleepers)
                self.ready.append(task)
                continue
            self._step(self.ready.popleft())
        return main.result

async def ticks(loop, n, interval):
    ___

async def rate_limited(loop, items, gap):
    ___

async def collect(agen):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_streams():\n    \"async generators\"\n    loop = Loop()\n    async def main():\n        return await collect(ticks(loop, 3, 2))\n    assert loop.run_until_complete(main()) == [2, 4, 6]\n    loop2 = Loop()\n    async def main2():\n        out = []\n        async for item in rate_limited(loop2, [\"a\", \"b\", \"c\"], 5):\n            out.append((item, loop2.now))\n        return out\n    assert loop2.run_until_complete(main2()) == [(\"a\", 0), (\"b\", 5), (\"c\", 10)]\n    loop3 = Loop()\n    async def slow_consumer():\n        out = []\n        async for item in rate_limited(loop3, [1, 2, 3], 2):\n            out.append((item, loop3.now))\n            await Sleep(3)\n        return out\n    assert loop3.run_until_complete(slow_consumer()) == [(1, 0), (2, 3), (3, 6)]")

### Bug hunt: Backoff that starts too high

The exponent starts at 1 instead of 0, so the first retry waits 2 units instead of 1, and every wait is doubled: this prints `[2, 4, 8]`. Start the exponent at 0 so it prints `[1, 2, 4]`.

In [ ]:
def backoff_delays(attempts, base=1, factor=2):
    return [base * factor ** i for i in range(1, attempts)]

print(backoff_delays(4))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"exponents from 0\"\n    expect_output(\"[1, 2, 4]\")")

### Bug hunt: A connection never closed

The code enters the session by calling `__aenter__` by hand, so when the work raises, nothing calls `__aexit__` and the log never records `close`: this prints `['open']`. Use `async with`, which closes it even after the error, so this prints `['open', 'close']`.

In [ ]:
class Pause:
    def __await__(self):
        yield

class Session:
    def __init__(self, log):
        self.log = log
    async def __aenter__(self):
        await Pause()
        self.log.append("open")
        return self
    async def __aexit__(self, *exc):
        self.log.append("close")
        return False

async def main(log):
    s = await Session(log).__aenter__()
    raise KeyError("lost")

log = []
coro = main(log)
try:
    while True:
        coro.send(None)
except (StopIteration, KeyError):
    pass
print(log)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"async with\"\n    expect_output(\"['open', 'close']\")")

### Bug hunt: Results in the wrong order

This "gather" collects results in the order the jobs **finished**, but callers expect them in the order the jobs were **given** (as `asyncio.gather` returns them), so this prints `['fast', 'mid', 'slow']`. Return them in input order so it prints `['slow', 'fast', 'mid']`.

In [ ]:
jobs = [("slow", 5), ("fast", 1), ("mid", 3)]
finished = sorted(jobs, key=lambda job: job[1])
print([name for name, _ in finished])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"input order\"\n    expect_output(\"['slow', 'fast', 'mid']\")")

### Puzzle: Fetching with a limit

Arrange the lines of a coroutine that fetches every URL with at most five requests in flight (real `asyncio`). One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # waits before attempts 2, 3 and 4 with base 1 and factor 2, as a list
q = ___   # how many permits asyncio.Lock has
r = ___   # the methods async with calls, as a tuple of two names
s = ___   # how many None sentinels stop 3 queue workers
t = ___   # True or False: async for works on an ordinary generator

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [1, 2, 4] and q == 1 and r == (\"__aenter__\", \"__aexit__\"), \"p, q and r\"\n    assert s == 3 and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: limiting concurrency

Opening a thousand connections at once can overwhelm a server or your own machine, so real async programs **limit concurrency** with a semaphore: `async with sem:` around the work. Using the loop in the starter, write `AsyncSemaphore(loop, value)`, a `Waitable` with a coroutine `acquire()` (waiting while no permits are left), `release()` (returning a permit and waking one waiter), `async with` support (released even if the block raises), and counters `active` and `peak` (the most permits held at once). Then the coroutine `bounded_gather(loop, coros, limit)`: run the coroutines concurrently with at most `limit` running at a time, returning `(results in the order given, peak)`.

In [ ]:
import heapq
from collections import deque

class Sleep:
    def __init__(self, seconds):
        self.seconds = seconds
    def __await__(self):
        yield ("sleep", self.seconds)

class Waitable:
    "Anything tasks can wait on: the loop adds the waiting task to .waiters."
    def __init__(self):
        self.waiters = deque()
    def wait(self):
        return _Wait(self)

class _Wait:
    def __init__(self, target):
        self.target = target
    def __await__(self):
        yield ("wait", self.target)

class Task(Waitable):
    def __init__(self, coro):
        super().__init__()
        self.coro, self.done, self.result = coro, False, None
    def __await__(self):
        if not self.done:
            yield ("wait", self)
        return self.result

class Loop:
    def __init__(self):
        self.now, self.ready, self.sleepers, self._seq = 0, deque(), [], 0
    def create_task(self, coro):
        task = Task(coro)
        self.ready.append(task)
        return task
    def wake(self, waitable, count=None):
        "Move waiting tasks (all, or up to count) back to the ready queue."
        while waitable.waiters and (count is None or count > 0):
            self.ready.append(waitable.waiters.popleft())
            if count is not None:
                count -= 1
    def _step(self, task):
        try:
            kind, arg = task.coro.send(None)
        except StopIteration as stop:
            task.done, task.result = True, stop.value
            self.wake(task)
            return
        if kind == "sleep":
            self._seq += 1
            heapq.heappush(self.sleepers, (self.now + arg, self._seq, task))
        elif kind == "wait":
            arg.waiters.append(task)
    def run_until_complete(self, coro):
        main = self.create_task(coro)
        while self.ready or self.sleepers:
            if not self.ready:
                self.now, _, task = heapq.heappop(self.sleepers)
                self.ready.append(task)
                continue
            self._step(self.ready.popleft())
        return main.result

class AsyncSemaphore(Waitable):
    def __init__(self, loop, value):
        ___

async def bounded_gather(loop, coros, limit):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_bounded():\n    \"a semaphore\"\n    loop = Loop()\n    starts = []\n    async def job(name, delay):\n        starts.append((name, loop.now))\n        await Sleep(delay)\n        return name\n    async def main():\n        return await bounded_gather(loop, [job(\"a\", 4), job(\"b\", 1), job(\"c\", 2), job(\"d\", 1)], limit=2)\n    results, peak = loop.run_until_complete(main())\n    assert results == [\"a\", \"b\", \"c\", \"d\"] and peak == 2 and loop.now == 4\n    assert starts == [(\"a\", 0), (\"b\", 0), (\"c\", 1), (\"d\", 3)]\n    loop2 = Loop()\n    async def main2():\n        return await bounded_gather(loop2, [job(str(i), 1) for i in range(6)], limit=3)\n    assert loop2.run_until_complete(main2())[1] == 3 and loop2.now == 2\n    loop3 = Loop()\n    sem = AsyncSemaphore(loop3, 1)\n    async def boom():\n        async with sem:\n            raise ValueError\n    async def main3():\n        try:\n            await boom()\n        except ValueError:\n            pass\n        async with sem:\n            return sem.value\n    assert loop3.run_until_complete(main3()) == 0 and sem.value == 1")

## Warm-up

With no limit, six two-second jobs finish together at 2; with at most two at a time, they finish in pairs, at 6. Limiting concurrency trades some speed for not overwhelming a server (or your own machine), and choosing the limit is a design decision.

## Limiting concurrency

`gather` with a thousand coroutines starts a thousand requests at once. Servers often refuse that (and may block you), and your machine may run out of sockets or file handles. **`asyncio.Semaphore(n)`** lets at most `n` tasks into a section at a time:

```
import asyncio

async def fetch(session, url, limit):
    async with limit:                       # waits while n tasks are inside
        return await session.get(url)

async def main(urls):
    limit = asyncio.Semaphore(10)
    return await asyncio.gather(*(fetch(session, u, limit) for u in urls))
```

The `async with` releases the permit even if the request fails. (`asyncio.Lock` is the special case `n = 1`, for protecting state across an `await`.)

## Queues between tasks

**`asyncio.Queue`** is the async version of `queue.Queue`: `await q.put(item)` (waiting if a `maxsize` is reached), `await q.get()` (waiting while it's empty), `q.task_done()` and `await q.join()`. It's the natural way to build **pipelines**: producer tasks put work, a fixed pool of consumer tasks takes it, and the pool size is the concurrency limit.

```
async def worker(queue, results):
    while True:
        url = await queue.get()
        if url is None:                      # a sentinel: stop
            return
        results.append(await fetch(url))

async def main(urls, n=5):
    queue, results = asyncio.Queue(), []
    workers = [asyncio.create_task(worker(queue, results)) for _ in range(n)]
    for url in urls:
        await queue.put(url)
    for _ in workers:
        await queue.put(None)
    await asyncio.gather(*workers)
    return results
```

Inside, a queue is a list of items plus a list of tasks waiting for an item; `put` adds an item and wakes **one** waiter. The exercises build exactly that.

## Timeouts and retries

Network operations can hang. Always put a **deadline** on them:

- `await asyncio.wait_for(coro, timeout=5)` cancels the operation and raises `TimeoutError` if it takes longer than 5 seconds (yesterday's challenge built one);
- `async with asyncio.timeout(5):` (3.11+) puts a deadline on a whole block.

When an operation fails for a **temporary** reason (a timeout, a connection refused, a "try again later" reply), **retry** it, but not immediately and not for ever: wait with **exponential backoff** (1, 2, 4, 8 seconds, up to a cap), give up after a few attempts, and add a little **jitter** (randomness) to the waits so that many clients don't all retry at the same moment. Never retry operations that aren't safe to repeat (such as charging a payment) unless the server guarantees it's idempotent.

```
async def fetch_with_retry(url, attempts=4):
    for i in range(attempts):
        try:
            return await asyncio.wait_for(fetch(url), timeout=5)
        except (TimeoutError, ConnectionError):
            if i == attempts - 1:
                raise
            await asyncio.sleep(min(30, 2 ** i) + random.random())
```

## Async context managers

Some set-up and clean-up needs to **wait**: opening a connection, starting a transaction, closing politely. **`async with`** is the async version of Week 61's `with`: it awaits `__aenter__()` on entry and `__aexit__(exc_type, exc, tb)` on exit, which must both be coroutines. `contextlib.asynccontextmanager` turns an async generator with one `yield` into one, just as `@contextmanager` does.

```
from contextlib import asynccontextmanager

@asynccontextmanager
async def connected(host):
    conn = await open_connection(host)
    try:
        yield conn
    finally:
        await conn.close()

async def main():
    async with connected("db.local") as conn:
        rows = await conn.query("select ...")
```

## Streams

An **async generator** (`async def` with `yield`) produces values that take time to arrive: lines from a socket, pages of an API's results, messages from a queue. Consume one with **`async for`**, or an **async comprehension**, `[x async for x in stream]`. They compose like ordinary generators (Week 59): one async generator can filter or transform another.

```
async def pages(api):
    url = api.first_page
    while url:
        page = await api.get(url)
        for item in page.items:
            yield item
        url = page.next

async def main(api):
    async for item in pages(api):
        print(item)
```

## Practice

## Quiz

## Daily challenge

## Before you go

On your own computer, use real `asyncio` to run twenty `asyncio.sleep(1)` "downloads" with a `Semaphore(5)` and check that they take about four seconds. Tomorrow: the mistakes that make async programs slow, stuck or wrong.